## Step 0 — Install and Import Libraries

Before starting Milestone 10, ensure your environment has the required packages:

In [ ]:
# milestone10_multimodal_phenotypic_screening

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

# RDKit for molecules (used in your previous milestones)
from rdkit import Chem
from rdkit.Chem import rdchem


## Step 1 — Create a Paired Molecule–Image Dataset

This dataset stores:
- SMILES strings
- Image tensors  
It returns both together so the model can learn molecule ↔ phenotype alignment.

In [13]:
class MoleculeImageDataset(Dataset):
    """
    Beginner-friendly dummy dataset:
    - smiles_list: list of SMILES strings
    - image_tensors: list of pre-loaded image tensors (C x H x W)
    """

    def __init__(self, smiles_list, image_tensors):
        assert len(smiles_list) == len(image_tensors), "Molecule and image counts must match."
        self.smiles_list = smiles_list
        self.image_tensors = image_tensors

    def __len__(self):
        return len(self.smiles_list)

    def smiles_to_mol(self, smiles):
        """Convert SMILES to RDKit Mol object."""
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"Invalid SMILES: {smiles}")
        return mol

    def __getitem__(self, idx):
        smiles = self.smiles_list[idx]
        img = self.image_tensors[idx]

        mol = self.smiles_to_mol(smiles)

        # For now, we just return the RDKit Mol and image tensor.
        # Later we will convert Mol → graph tensors.
        return mol, img


## Step 2 — Build a Simple Molecular Encoder

This is a beginner-friendly encoder:
- Counts atoms by atomic number
- Converts the count vector into a 256‑dim embedding

Later you can replace this with your real GNN.

In [14]:
class SimpleMolEncoder(nn.Module):
    """
    Very simple molecular encoder:
    - Counts atoms by atomic number
    - Embeds the count vector into a fixed-size representation
    """

    def __init__(self, max_atomic_num=100, emb_dim=256):
        super().__init__()
        self.max_atomic_num = max_atomic_num
        self.emb_dim = emb_dim

        # Linear layer to map atom-count vector → embedding
        self.fc = nn.Linear(max_atomic_num, emb_dim)

    def forward(self, mol):
        # Create a simple atom-count vector
        atom_counts = torch.zeros(self.max_atomic_num, dtype=torch.float32)

        for atom in mol.GetAtoms():
            Z = atom.GetAtomicNum()
            if Z < self.max_atomic_num:
                atom_counts[Z] += 1.0

        # Add batch dimension: (1, max_atomic_num)
        atom_counts = atom_counts.unsqueeze(0)

        # Map to embedding
        emb = self.fc(atom_counts)  # (1, emb_dim)
        emb = F.relu(emb)
        return emb  # shape: (1, emb_dim)


## Step 3 — Build an Image Encoder Using ResNet18

This encoder:
- Loads pretrained ResNet18
- Replaces the final layer with a 256‑dim projection
- Outputs an image embedding

In [15]:
class ImageEncoder(nn.Module):
    """
    Image encoder using ResNet18:
    - Takes an image tensor (C x H x W)
    - Outputs a fixed-size embedding
    """

    def __init__(self, emb_dim=256):
        super().__init__()
        # Load a ResNet18 backbone
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Replace the final fully-connected layer with a new one
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, emb_dim)

    def forward(self, img):
        # Add batch dimension if needed: (C, H, W) → (1, C, H, W)
        if img.dim() == 3:
            img = img.unsqueeze(0)

        emb = self.backbone(img)  # (1, emb_dim)
        emb = F.relu(emb)
        return emb


## Step 4 — Build the Multimodal CLIP-Style Model

This model:
- Takes molecule + image embeddings
- Projects both into a shared 128‑dim space
- Computes CLIP-style contrastive loss

In [16]:
class MultimodalModel(nn.Module):
    """
    - Takes molecule embedding (256-d)
    - Takes image embedding (256-d)
    - Projects both into a shared 128-d space
    - Computes CLIP-style contrastive similarity
    """

    def __init__(self, emb_dim=256, proj_dim=128):
        super().__init__()

        # Projection heads for molecule and image
        self.mol_proj = nn.Linear(emb_dim, proj_dim)
        self.img_proj = nn.Linear(emb_dim, proj_dim)

    def forward(self, mol_emb, img_emb):
        """
        mol_emb: (batch, 256)
        img_emb: (batch, 256)
        """

        # Project into shared space
        z_mol = self.mol_proj(mol_emb)      # (batch, 128)
        z_img = self.img_proj(img_emb)      # (batch, 128)

        # Normalise (important for contrastive learning)
        z_mol = F.normalize(z_mol, dim=1)
        z_img = F.normalize(z_img, dim=1)

        return z_mol, z_img

    def contrastive_loss(self, z_mol, z_img, temperature=0.1):
        """
        CLIP-style InfoNCE loss:
        - Positive pairs: matching molecule ↔ image
        - Negatives: all other items in batch
        """

        # Similarity matrix (batch x batch)
        sim_matrix = torch.matmul(z_mol, z_img.T) / temperature

        # Labels: diagonal = correct matches
        batch_size = sim_matrix.size(0)
        labels = torch.arange(batch_size).to(sim_matrix.device)

        # Cross-entropy loss for both directions
        loss_mol_to_img = F.cross_entropy(sim_matrix, labels)
        loss_img_to_mol = F.cross_entropy(sim_matrix.T, labels)

        return (loss_mol_to_img + loss_img_to_mol) / 2


## Step 4.1 — Quick Test With Fake Embeddings

This verifies the model works before training.

In [17]:
mol_encoder = SimpleMolEncoder()
img_encoder = ImageEncoder()


In [18]:
mol_emb = torch.randn(1, 256)   # fake molecule embedding
img_emb = torch.randn(1, 256)   # fake image embedding


In [19]:
model = MultimodalModel()


In [20]:
z_mol, z_img = model(mol_emb, img_emb)


In [21]:
loss = model.contrastive_loss(z_mol, z_img)
print("Loss:", loss.item())


Loss: 0.0


In [22]:
mol_emb = torch.randn(4, 256)   # batch of 4 molecules
img_emb = torch.randn(4, 256)   # batch of 4 images

model = MultimodalModel()

z_mol, z_img = model(mol_emb, img_emb)
loss = model.contrastive_loss(z_mol, z_img)

print("Loss:", loss.item())


Loss: 1.51070237159729
